# XGBoost Recreation On Current Dataset

This notebook tries to recreate the older high XGBoost result using the current `data_clean/all_fights.csv` dataset.

Design choices:

- Uses random 90/10 validation splits, repeated 5 times.
- Balances `R` and `B` wins by randomly side-swapping rows until the labels are close to 50/50.
- Collapses rare stance categories into `Other` so the model cannot rely on tiny artifacts like `Sideways` stance.
- Tests both same-fight stat features and realistic pre-fight historical features.

The same-fight feature set is intentionally leaky. It exists only to test how high accuracy can get when the model sees fight-result stats.

In [2]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
SPLIT_SEEDS = [11, 22, 33, 44, 55]
TEST_SIZE = 0.10
MIN_STANCE_COUNT = 20

XGB_PARAMS = dict(
    n_estimators=700,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
)

DATA_PATH = Path("..") / "data_clean" / "all_fights.csv"

## Load And Basic Cleaning

In [3]:
def parse_height_inches(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.search(r"(\d+)\s*'\s*(\d+)", text)
    if match:
        return int(match.group(1)) * 12 + int(match.group(2))
    return pd.to_numeric(text, errors="coerce")

def parse_reach_inches(value):
    if pd.isna(value):
        return np.nan
    return pd.to_numeric(str(value).replace('"', '').strip(), errors="coerce")

def parse_time_seconds(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text or text == "--":
        return np.nan
    parts = text.split(":")
    if len(parts) != 2:
        return np.nan
    return int(parts[0]) * 60 + int(parts[1])

def normalize_rare_categories(series, min_count=20):
    filled = series.fillna("Unknown").astype(str).str.strip().replace("", "Unknown")
    counts = filled.value_counts()
    rare = counts[counts < min_count].index
    return filled.where(~filled.isin(rare), "Other")

raw = pd.read_csv(DATA_PATH)
df = raw.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["winner_side"] = np.select(
    [df["winner"].eq(df["R_fighter"]), df["winner"].eq(df["B_fighter"])],
    ["R", "B"],
    default="",
)
df["winner_side"] = df["winner_side"].replace("", np.nan)
df = df.dropna(subset=["date", "winner_side", "R_fighter", "B_fighter"]).copy()
df = df.sort_values(["date", "R_fighter", "B_fighter"]).reset_index(drop=True)
df["fight_id"] = np.arange(len(df))

stat_suffixes = [
    "KD", "SIG_STR_landed", "SIG_STR_att", "SIG_STR_pct", "TOTAL_STR_landed", "TOTAL_STR_att",
    "TD_landed", "TD_att", "TD_pct", "SUB_ATT", "REV", "HEAD_landed", "HEAD_att",
    "BODY_landed", "BODY_att", "LEG_landed", "LEG_att", "DISTANCE_landed", "DISTANCE_att",
    "CLINCH_landed", "CLINCH_att", "GROUND_landed", "GROUND_att", "CTRL_seconds",
]

for side in ["R", "B"]:
    df[f"{side}_DOB_dt"] = pd.to_datetime(df[f"{side}_DOB"], errors="coerce")
    df[f"{side}_age"] = (df["date"] - df[f"{side}_DOB_dt"]).dt.days / 365.25
    df[f"{side}_Height_inches"] = df[f"{side}_Height"].map(parse_height_inches)
    df[f"{side}_Reach_inches"] = df[f"{side}_Reach"].map(parse_reach_inches)
    df[f"{side}_CTRL_seconds"] = df[f"{side}_CTRL"].map(parse_time_seconds)
    df[f"{side}_Stance_clean"] = normalize_rare_categories(df[f"{side}_Stance"], MIN_STANCE_COUNT)
    for suffix in stat_suffixes:
        col = f"{side}_{suffix}"
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

df["DIFF_age"] = df["R_age"] - df["B_age"]
df["DIFF_height"] = df["R_Height_inches"] - df["B_Height_inches"]
df["DIFF_reach"] = df["R_Reach_inches"] - df["B_Reach_inches"]
df["stance_matchup"] = df["R_Stance_clean"] + "_vs_" + df["B_Stance_clean"]

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
print(df["winner_side"].value_counts())
print("Majority baseline:", df["winner_side"].value_counts(normalize=True).max().round(4))
print("\nClean R stance counts:")
print(df["R_Stance_clean"].value_counts())
print("\nClean B stance counts:")
print(df["B_Stance_clean"].value_counts())

C:\Users\Abtin\AppData\Local\Temp\ipykernel_12200\4126237748.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[f"{side}_DOB_dt"] = pd.to_datetime(df[f"{side}_DOB"], errors="coerce")
C:\Users\Abtin\AppData\Local\Temp\ipykernel_12200\4126237748.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[f"{side}_DOB_dt"] = pd.to_datetime(df[f"{side}_DOB"], errors="coerce")


Loaded: ..\data_clean\all_fights.csv
Shape: (8549, 82)
winner_side
R    5477
B    3072
Name: count, dtype: int64
Majority baseline: 0.6407

Clean R stance counts:
R_Stance_clean
Orthodox    6334
Southpaw    1693
Switch       484
Unknown       21
Other         17
Name: count, dtype: int64

Clean B stance counts:
B_Stance_clean
Orthodox    6364
Southpaw    1618
Switch       497
Unknown       57
Other         13
Name: count, dtype: int64


## Random Side Balancing

This keeps the row count fixed. It randomly chooses enough rows from the majority winner side, swaps all `R_` and `B_` columns, and flips the target.

In [4]:
def swap_r_b_columns(data, rows_to_swap):
    swapped = data.copy()
    r_cols = [c for c in swapped.columns if c.startswith("R_")]
    for r_col in r_cols:
        suffix = r_col[2:]
        b_col = f"B_{suffix}"
        if b_col in swapped.columns:
            r_values = swapped.loc[rows_to_swap, r_col].copy()
            swapped.loc[rows_to_swap, r_col] = swapped.loc[rows_to_swap, b_col]
            swapped.loc[rows_to_swap, b_col] = r_values
    return swapped

def balance_by_random_side_swap(data, seed=42, target_col="winner_side"):
    balanced = data.copy()
    counts = balanced[target_col].value_counts()
    majority = counts.idxmax()
    minority = counts.idxmin()
    n_to_flip = int((counts[majority] - counts[minority]) // 2)
    rng = np.random.default_rng(seed)
    candidate_idx = balanced.index[balanced[target_col] == majority].to_numpy()
    rows_to_swap = rng.choice(candidate_idx, size=n_to_flip, replace=False)
    balanced = swap_r_b_columns(balanced, rows_to_swap)
    balanced.loc[rows_to_swap, target_col] = minority
    balanced["was_side_swapped"] = False
    balanced.loc[rows_to_swap, "was_side_swapped"] = True
    return balanced

balanced_df = balance_by_random_side_swap(df, seed=RANDOM_STATE)

# Rebuild difference features after swapping.
balanced_df["DIFF_age"] = balanced_df["R_age"] - balanced_df["B_age"]
balanced_df["DIFF_height"] = balanced_df["R_Height_inches"] - balanced_df["B_Height_inches"]
balanced_df["DIFF_reach"] = balanced_df["R_Reach_inches"] - balanced_df["B_Reach_inches"]
balanced_df["stance_matchup"] = balanced_df["R_Stance_clean"] + "_vs_" + balanced_df["B_Stance_clean"]

print("Original distribution:")
print(df["winner_side"].value_counts())
print("\nBalanced distribution:")
print(balanced_df["winner_side"].value_counts())
print("\nSwapped rows:", int(balanced_df["was_side_swapped"].sum()))

Original distribution:
winner_side
R    5477
B    3072
Name: count, dtype: int64

Balanced distribution:
winner_side
R    4275
B    4274
Name: count, dtype: int64

Swapped rows: 1202


## Modeling Helpers

In [5]:
def make_xy(feature_frame, target):
    X = feature_frame.copy()
    X = pd.get_dummies(X, dummy_na=True)
    X = X.replace([np.inf, -np.inf], np.nan)
    y = target.map({"B": 0, "R": 1}).astype(int)
    return X, y

def align_train_test(X_train, X_test):
    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)
    return X_train, X_test

def evaluate_xgb_five_splits(X, y, label, params=None):
    params = dict(XGB_PARAMS if params is None else params)
    rows = []
    last_model = None
    last_confusion = None
    last_importances = None

    for split_num, seed in enumerate(SPLIT_SEEDS, start=1):
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        train_idx, test_idx = next(splitter.split(X, y))
        X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

        X_train, X_test = align_train_test(X_train, X_test)

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(random_state=seed, **params)),
        ])
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        rows.append({
            "feature_set": label,
            "split": split_num,
            "seed": seed,
            "accuracy": accuracy_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "train_rows": len(train_idx),
            "test_rows": len(test_idx),
        })

        last_model = model
        last_confusion = confusion_matrix(y_test, pred)
        booster = model.named_steps["model"]
        last_importances = pd.Series(booster.feature_importances_, index=X_train.columns).sort_values(ascending=False)

    split_results = pd.DataFrame(rows)
    summary = pd.DataFrame([{
        "feature_set": label,
        "n_features": X.shape[1],
        "mean_accuracy": split_results["accuracy"].mean(),
        "std_accuracy": split_results["accuracy"].std(),
        "mean_auc": split_results["roc_auc"].mean(),
        "std_auc": split_results["roc_auc"].std(),
    }])
    return summary, split_results, last_confusion, last_importances

## Feature Set 1: Same-fight Stats

This is the high-accuracy but leaky version. It uses stats from the fight itself.

In [6]:
same_fight_cols = []
for side in ["R", "B"]:
    same_fight_cols.extend([
        f"{side}_age", f"{side}_Height_inches", f"{side}_Reach_inches", f"{side}_Stance_clean",
    ])
    same_fight_cols.extend([f"{side}_{suffix}" for suffix in stat_suffixes if f"{side}_{suffix}" in balanced_df.columns])

same_fight_cols.extend(["DIFF_age", "DIFF_height", "DIFF_reach", "stance_matchup"])

X_same, y_same = make_xy(balanced_df[same_fight_cols], balanced_df["winner_side"])
same_summary, same_splits, same_confusion, same_importances = evaluate_xgb_five_splits(
    X_same,
    y_same,
    "same-fight stats, balanced side swap",
)

display(same_summary)
display(same_splits)
print("Last split confusion matrix:")
print(same_confusion)
display(same_importances.head(30).to_frame("importance"))

,feature_set,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc
0,"same-fight stats, balanced side swap",90,0.893801,0.006939,0.960466,0.004699


,feature_set,split,seed,accuracy,roc_auc,train_rows,test_rows
0,"same-fight stats, balanced side swap",1,11,0.883041,0.952762,7694,855
1,"same-fight stats, balanced side swap",2,22,0.891228,0.960171,7694,855
2,"same-fight stats, balanced side swap",3,33,0.898246,0.964636,7694,855
3,"same-fight stats, balanced side swap",4,44,0.895906,0.963826,7694,855
4,"same-fight stats, balanced side swap",5,55,0.900585,0.960937,7694,855


Last split confusion matrix:
[[396  31]
 [ 54 374]]


,importance
R_GROUND_landed,0.172132
B_GROUND_landed,0.099523
B_KD,0.057512
R_KD,0.052362
B_TOTAL_STR_landed,0.045048
R_GROUND_att,0.038137
B_GROUND_att,0.035197
B_SUB_ATT,0.032314
B_SIG_STR_pct,0.030012
R_SUB_ATT,0.028082


## Feature Set 2: Same-fight Stats Without Stance

This checks whether high accuracy depends on stance artifacts.

In [7]:
no_stance_cols = [
    c for c in same_fight_cols
    if "Stance" not in c and c != "stance_matchup"
]

X_same_no_stance, y_same_no_stance = make_xy(balanced_df[no_stance_cols], balanced_df["winner_side"])
same_no_stance_summary, same_no_stance_splits, same_no_stance_confusion, same_no_stance_importances = evaluate_xgb_five_splits(
    X_same_no_stance,
    y_same_no_stance,
    "same-fight stats, no stance",
)

display(same_no_stance_summary)
display(same_no_stance_splits)
print("Last split confusion matrix:")
print(same_no_stance_confusion)
display(same_no_stance_importances.head(30).to_frame("importance"))

,feature_set,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc
0,"same-fight stats, no stance",57,0.894971,0.008491,0.959948,0.005013


,feature_set,split,seed,accuracy,roc_auc,train_rows,test_rows
0,"same-fight stats, no stance",1,11,0.880702,0.951969,7694,855
1,"same-fight stats, no stance",2,22,0.895906,0.958951,7694,855
2,"same-fight stats, no stance",3,33,0.902924,0.964056,7694,855
3,"same-fight stats, no stance",4,44,0.895906,0.964269,7694,855
4,"same-fight stats, no stance",5,55,0.899415,0.960494,7694,855


Last split confusion matrix:
[[395  32]
 [ 54 374]]


,importance
R_GROUND_landed,0.171590
B_GROUND_landed,0.109921
B_KD,0.065356
R_KD,0.057304
R_GROUND_att,0.047450
B_TOTAL_STR_landed,0.041954
B_SUB_ATT,0.033305
B_SIG_STR_pct,0.032080
R_TOTAL_STR_landed,0.031775
R_SUB_ATT,0.030286


## Feature Set 3: Historical Pre-fight Features

This reconstructs current-form, experience, win-rate, streak, and historical average stat features using only fights before the current fight.

In [8]:
def build_long_rows(data):
    rows = []
    for _, fight in data.sort_values(["date", "fight_id"]).iterrows():
        for side, opp in [("R", "B"), ("B", "R")]:
            row = {
                "fight_id": fight["fight_id"],
                "date": fight["date"],
                "fighter": fight[f"{side}_fighter"],
                "side": side,
                "won": int(fight["winner_side"] == side),
                "age": fight[f"{side}_age"],
                "height": fight[f"{side}_Height_inches"],
                "reach": fight[f"{side}_Reach_inches"],
                "stance": fight[f"{side}_Stance_clean"],
            }
            for suffix in stat_suffixes:
                side_col = f"{side}_{suffix}"
                opp_col = f"{opp}_{suffix}"
                if side_col in data.columns:
                    row[suffix] = fight[side_col]
                if opp_col in data.columns:
                    row[f"opp_{suffix}"] = fight[opp_col]
            rows.append(row)
    return pd.DataFrame(rows)

def add_pre_fight_history(long):
    long = long.sort_values(["date", "fight_id", "side"]).reset_index(drop=True).copy()
    history_rows = []
    stat_cols = [c for c in long.columns if c in stat_suffixes or c.startswith("opp_")]

    for fighter, group in long.groupby("fighter", sort=False):
        prior_fights = 0
        prior_wins = 0
        current_win_streak = 0
        current_lose_streak = 0
        longest_win_streak = 0
        sums = {c: 0.0 for c in stat_cols}

        for idx, row in group.iterrows():
            out = {
                "fight_id": row["fight_id"],
                "side": row["side"],
                "prior_fights": prior_fights,
                "prior_wins": prior_wins,
                "prior_losses": prior_fights - prior_wins,
                "prior_win_pct": prior_wins / prior_fights if prior_fights else np.nan,
                "current_win_streak": current_win_streak,
                "current_lose_streak": current_lose_streak,
                "longest_win_streak": longest_win_streak,
                "age": row["age"],
                "height": row["height"],
                "reach": row["reach"],
                "stance": row["stance"],
            }
            for c in stat_cols:
                out[f"hst_{c}_avg"] = sums[c] / prior_fights if prior_fights else np.nan
            history_rows.append(out)

            won = int(row["won"])
            prior_fights += 1
            prior_wins += won
            if won:
                current_win_streak += 1
                current_lose_streak = 0
                longest_win_streak = max(longest_win_streak, current_win_streak)
            else:
                current_lose_streak += 1
                current_win_streak = 0
            for c in stat_cols:
                value = pd.to_numeric(row[c], errors="coerce")
                sums[c] += 0.0 if pd.isna(value) else float(value)

    return pd.DataFrame(history_rows)

long = build_long_rows(balanced_df)
hist = add_pre_fight_history(long)

hist_R = hist[hist["side"] == "R"].drop(columns="side").add_prefix("R_").rename(columns={"R_fight_id": "fight_id"})
hist_B = hist[hist["side"] == "B"].drop(columns="side").add_prefix("B_").rename(columns={"B_fight_id": "fight_id"})

pre = balanced_df[["fight_id", "winner_side"]].merge(hist_R, on="fight_id", how="left").merge(hist_B, on="fight_id", how="left")

base_diff_pairs = [
    "prior_fights", "prior_wins", "prior_losses", "prior_win_pct", "current_win_streak",
    "current_lose_streak", "longest_win_streak", "age", "height", "reach",
]
hist_avg_cols = [c[6:] for c in pre.columns if c.startswith("R_hst_")]
for suffix in base_diff_pairs + hist_avg_cols:
    r_col = f"R_{suffix}"
    b_col = f"B_{suffix}"
    if r_col in pre.columns and b_col in pre.columns:
        pre[f"DIFF_{suffix}"] = pd.to_numeric(pre[r_col], errors="coerce") - pd.to_numeric(pre[b_col], errors="coerce")

pre["stance_matchup"] = pre["R_stance"].fillna("Unknown") + "_vs_" + pre["B_stance"].fillna("Unknown")

historical_cols = [c for c in pre.columns if c not in ["fight_id", "winner_side"]]
X_hist, y_hist = make_xy(pre[historical_cols], pre["winner_side"])

hist_summary, hist_splits, hist_confusion, hist_importances = evaluate_xgb_five_splits(
    X_hist,
    y_hist,
    "historical pre-fight features",
)

display(hist_summary)
display(hist_splits)
print("Last split confusion matrix:")
print(hist_confusion)
display(hist_importances.head(30).to_frame("importance"))

,feature_set,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc
0,historical pre-fight features,159,0.631111,0.018393,0.684624,0.017358


,feature_set,split,seed,accuracy,roc_auc,train_rows,test_rows
0,historical pre-fight features,1,11,0.653801,0.697892,7694,855
1,historical pre-fight features,2,22,0.610526,0.655015,7694,855
2,historical pre-fight features,3,33,0.636257,0.694117,7694,855
3,historical pre-fight features,4,44,0.614035,0.683638,7694,855
4,historical pre-fight features,5,55,0.640936,0.692459,7694,855


Last split confusion matrix:
[[280 147]
 [160 268]]


,importance
R_hst_opp_SIG_STR_landed_avg,0.030147
R_hst_opp_HEAD_landed_avg,0.017397
DIFF_age,0.016669
B_hst_SIG_STR_att_avg,0.015083
B_hst_SIG_STR_landed_avg,0.014996
DIFF_longest_win_streak,0.014992
B_reach,0.013097
DIFF_prior_win_pct,0.012299
B_hst_DISTANCE_landed_avg,0.012181
B_prior_wins,0.012060


## Combined Comparison

In [9]:
summary = pd.concat([same_summary, same_no_stance_summary, hist_summary], ignore_index=True)
summary = summary.sort_values("mean_accuracy", ascending=False)
display(summary)

splits = pd.concat([same_splits, same_no_stance_splits, hist_splits], ignore_index=True)
display(splits.pivot(index="split", columns="feature_set", values="accuracy"))

,feature_set,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc
1,"same-fight stats, no stance",57,0.894971,0.008491,0.959948,0.005013
0,"same-fight stats, balanced side swap",90,0.893801,0.006939,0.960466,0.004699
2,historical pre-fight features,159,0.631111,0.018393,0.684624,0.017358


feature_set,historical pre-fight features,"same-fight stats, balanced side swap","same-fight stats, no stance"
split,,,
1,0.653801,0.883041,0.880702
2,0.610526,0.891228,0.895906
3,0.636257,0.898246,0.902924
4,0.614035,0.895906,0.895906
5,0.640936,0.900585,0.899415
